<a href="https://colab.research.google.com/github/nosadchiy/public/blob/main/Forecasting_RQ_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [219]:
# Vibe coded by Nikolay Osadchiy, (c) 2026
# Calls tools agentically to compute Reorder point and Order quantity

!pip -q install openai pandas numpy scipy

import os, json, uuid
import numpy as np
import pandas as pd
from scipy.stats import norm
from openai import OpenAI
from google.colab import userdata

# Securely retrieve the key from Colab Secrets
# Make sure you have added 'OPENAI_API_KEY' to the Secrets tab
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [220]:
DATASETS = {}

def register_dataset(df: pd.DataFrame) -> str:
    dsid = uuid.uuid4().hex[:8]
    DATASETS[dsid] = df.copy()
    return dsid

In [221]:
def profile_demand_series(dataset_id: str, date_col: str, demand_col: str, freq: str = "D") -> dict:
    df = DATASETS[dataset_id].copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)

    y = df[demand_col].astype(float).values
    missing = int(pd.isna(df[demand_col]).sum())

    # crude seasonality hint: compare variance of lagged differences
    dif1 = np.diff(y) if len(y) > 2 else np.array([0.0])
    dif7 = (y[7:] - y[:-7]) if len(y) > 10 else np.array([0.0])

    return {
        "n_obs": int(len(y)),
        "missing": missing,
        "mean": float(np.nanmean(y)),
        "std": float(np.nanstd(y, ddof=1)) if len(y) > 1 else 0.0,
        "min": float(np.nanmin(y)),
        "max": float(np.nanmax(y)),
        "seasonality_hint": "weekly" if np.nanvar(dif7) < np.nanvar(dif1) else "none_or_unknown",
        "last_7": [float(v) for v in y[-7:]] if len(y) >= 7 else [float(v) for v in y],
        "freq": freq
    }

In [222]:
def forecast_baseline(
    dataset_id: str,
    date_col: str,
    demand_col: str,
    horizon: int,
    method: str = "naive",          # "naive" | "seasonal_naive" | "moving_average"
    seasonal_period: int = 7,       # for seasonal_naive
    ma_window: int = 7              # for moving_average
) -> dict:
    df = DATASETS[dataset_id].copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col)
    y = df[demand_col].astype(float).values

    if len(y) == 0:
        raise ValueError("Empty series.")

    if method == "naive":
        fc = np.repeat(y[-1], horizon)

    elif method == "seasonal_naive":
        if len(y) < seasonal_period:
            fc = np.repeat(y[-1], horizon)
        else:
            season = y[-seasonal_period:]
            reps = int(np.ceil(horizon / seasonal_period))
            fc = np.tile(season, reps)[:horizon]

    elif method == "moving_average":
        w = min(ma_window, len(y))
        level = float(np.mean(y[-w:]))
        fc = np.repeat(level, horizon)

    else:
        raise ValueError(f"Unknown method: {method}")

    # quick backtest: last horizon as test if possible
    if len(y) > horizon:
        train, test = y[:-horizon], y[-horizon:]
        # Use register_dataset to get a UNIQUE ID for this recursion level
        tmp_df = pd.DataFrame({"t": np.arange(len(train)), "y": train})
        tmp_id = register_dataset(tmp_df)

        # Recursive call with a unique tmp_id instead of a hardcoded string
        bt = forecast_baseline(tmp_id, "t", "y", horizon, method, seasonal_period, ma_window)

        # Clean up the specific temporary entry
        if tmp_id in DATASETS:
            del DATASETS[tmp_id]

        pred = np.array(bt["forecast"])
        mape = float(np.mean(np.abs((test - pred) / np.maximum(1e-9, np.abs(test)))) * 100.0)
    else:
        mape = None

    return {
        "method": method,
        "horizon": int(horizon),
        "forecast": [float(v) for v in fc],
        "backtest_mape_percent": mape
    }

In [223]:
def compute_rq_policy(
    mean_demand_per_period: float,
    std_demand_per_period: float,
    lead_time_periods: float,
    service_level: float,          # e.g., 0.95
    ordering_cost: float,          # K
    holding_cost_per_unit_period: float  # h
) -> dict:
    if not (0.5 < service_level < 0.9999):
        raise ValueError("service_level should be between ~0.5 and 0.9999")

    mu_L = mean_demand_per_period * lead_time_periods
    sigma_L = std_demand_per_period * np.sqrt(lead_time_periods)

    z = float(norm.ppf(service_level))
    safety_stock = z * sigma_L
    reorder_point = mu_L + safety_stock

    # EOQ
    D = mean_demand_per_period  # per period demand rate (for EOQ scaling, think "period" as your time unit)
    if holding_cost_per_unit_period <= 0:
        raise ValueError("holding_cost must be > 0")
    Q = float(np.sqrt(2.0 * ordering_cost * max(D, 1e-9) / holding_cost_per_unit_period))

    return {
        "assumption": "Normal lead-time demand; continuous review (R,Q) approximation",
        "z": z,
        "mu_leadtime": float(mu_L),
        "sigma_leadtime": float(sigma_L),
        "safety_stock": float(safety_stock),
        "reorder_point": float(reorder_point),
        "eoq_Q": float(Q)
    }

In [224]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "profile_demand_series",
            "description": "Profile a time series of demand and return basic stats + a seasonality hint.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_id": {"type": "string"},
                    "date_col": {"type": "string"},
                    "demand_col": {"type": "string"},
                    "freq": {"type": "string", "description": "Pandas-like frequency label, e.g. 'D' or 'W'."}
                },
                "required": ["dataset_id", "date_col", "demand_col"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "forecast_baseline",
            "description": "Create a simple baseline forecast (naive, seasonal_naive, moving_average) and optional backtest MAPE.",
            "parameters": {
                "type": "object",
                "properties": {
                    "dataset_id": {"type": "string"},
                    "date_col": {"type": "string"},
                    "demand_col": {"type": "string"},
                    "horizon": {"type": "integer", "minimum": 1},
                    "method": {"type": "string", "enum": ["naive", "seasonal_naive", "moving_average"]},
                    "seasonal_period": {"type": "integer", "minimum": 2},
                    "ma_window": {"type": "integer", "minimum": 2}
                },
                "required": ["dataset_id", "date_col", "demand_col", "horizon"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compute_EOQ_ReorderPoint",
            "description": "Compute EOQ order quantity and reorder point R with safety stock under Normal lead-time demand assumption.",
            "parameters": {
                "type": "object",
                "properties": {
                    "mean_demand_per_period": {"type": "number"},
                    "std_demand_per_period": {"type": "number", "minimum": 0},
                    "lead_time_periods": {"type": "number", "minimum": 0},
                    "service_level": {"type": "number"},
                    "ordering_cost": {"type": "number", "minimum": 0},
                    "holding_cost_per_unit_period": {"type": "number", "minimum": 0}
                },
                "required": ["mean_demand_per_period", "std_demand_per_period", "lead_time_periods",
                             "service_level", "ordering_cost", "holding_cost_per_unit_period"]
            }
        }
    }
]

In [225]:
TOOL_IMPL = {
    "profile_demand_series": profile_demand_series,
    "forecast_baseline": forecast_baseline,
    "compute_EOQ_ReorderPoint": compute_rq_policy
}

In [226]:
SYSTEM_INSTRUCTIONS =  f"""
You are a Supply Chain Analytics Coach.
Rules: Never invent numeric results. If you need numbers, call a tool.
If you need EOQ or Reorder point, call the respective tool.
Always include: (1) Assumptions, (2) Computation summary, (3) Recommendation, (4) Sanity checks.
Prefer simple baselines first (naive/seasonal_naive) before fancy methods.
"""

def run_agent(user_message: str, model: str = "gpt-5.4-mini"):
    input_list = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": user_message},
    ]

    print("USER MESSAGE:", user_message)

    while True:
        response = client.chat.completions.create(
            model=model,
            tools=TOOLS,
            messages=input_list,
        )

        response_message = response.choices[0].message

        if not response_message.tool_calls:
            print("FINAL RESPONSE:", response_message.content)
            return response_message.content

        print("RESPONSE:", response_message.content)
        input_list.append(response_message)

        for tool_call in response_message.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            print(f"Executing Tool: {fn_name} with {args}")
            if fn_name in TOOL_IMPL:
                result = TOOL_IMPL[fn_name](**args)
            else:
                result = {"error": f"Tool {fn_name} not found."}

            input_list.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "name": fn_name,
                "content": json.dumps(result)
            })

# Demo

In [227]:
# Example dataset with Normal distribution
df = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=60, freq="D"),
    "demand": np.random.normal(loc=20, scale=5, size=60).clip(0)
})
dsid = register_dataset(df)

print(df.head())
print(f"\nActual mean: {df['demand'].mean():.2f}, Actual std: {df['demand'].std():.2f}")

        date     demand
0 2024-01-01  19.078307
1 2024-01-02  16.137608
2 2024-01-03  13.157396
3 2024-01-04  18.999313
4 2024-01-05  21.518079

Actual mean: 19.31, Actual std: 4.43


In [228]:
prompt = f"""
We sell one SKU. Here is the dataset_id={dsid} with columns date,demand.
1) Profile demand and pick a baseline forecast for horizon=14 days.
2) Using the estimated mean/std per day from the profile, compute the reorder point and EOQ quantity with:
   lead_time=5 days, service_level=0.95, ordering_cost K=200, holding_cost h=0.1 per unit per day.
Explain assumptions and provide sanity checks.
"""

print(run_agent(prompt))

USER MESSAGE: 
We sell one SKU. Here is the dataset_id=005fd567 with columns date,demand.
1) Profile demand and pick a baseline forecast for horizon=14 days.
2) Using the estimated mean/std per day from the profile, compute the reorder point and EOQ quantity with:
   lead_time=5 days, service_level=0.95, ordering_cost K=200, holding_cost h=0.1 per unit per day.
Explain assumptions and provide sanity checks.

RESPONSE: None
Executing Tool: profile_demand_series with {'dataset_id': '005fd567', 'date_col': 'date', 'demand_col': 'demand', 'freq': 'D'}
Executing Tool: forecast_baseline with {'dataset_id': '005fd567', 'date_col': 'date', 'demand_col': 'demand', 'horizon': 14, 'method': 'seasonal_naive', 'seasonal_period': 7}
Executing Tool: forecast_baseline with {'dataset_id': '005fd567', 'date_col': 'date', 'demand_col': 'demand', 'horizon': 14, 'method': 'naive'}
Executing Tool: compute_EOQ_ReorderPoint with {'mean_demand_per_period': 0, 'std_demand_per_period': 0, 'lead_time_periods': 5,